# Router Unit Test Notebook

This notebook tests the Artemis Router inference system:

1. Load configuration and initialize RouterEngine
2. Test with synthetic samples
3. Test with database samples (single)
4. Test with database samples (batch)
5. Verify logging and model selection

**Prerequisites:**
- Trained router checkpoint available
- Database populated with test samples
- Configuration file created and paths updated

In [ ]:
# Imports
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from artemis_router.config import load_config
from artemis_router.router_engine import RouterEngine
from artemis_router.traffic_simulator import make_synthetic_sample

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("Imports successful!")

## 1. Load Configuration

In [ ]:
# Update this path to your actual config file
config_path = "../router_config_example.yaml"

# Load configuration
cfg = load_config(config_path)

print("Configuration loaded successfully!")
print(f"\nRouter device: {cfg.router.device}")
print(f"Router dtype: {cfg.router.dtype}")
print(f"Number of models: {len(cfg.router.model_name_order)}")
print(f"Model order: {cfg.router.model_name_order}")
print(f"\nSQL logging: {cfg.logging.sql_enabled}")
print(f"W&B logging: {cfg.logging.wandb_enabled}")
print(f"LB enabled: {cfg.lb.enabled}")

## 2. Initialize Router Engine

In [ ]:
# Initialize engine (this will load the model and run warmup)
engine = RouterEngine(cfg)

print("\nRouter engine initialized!")
print("\nEngine stats:")
stats = engine.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

## 3. Test with Synthetic Sample

In [ ]:
# Create a synthetic sample
sample = make_synthetic_sample(0, cfg.traffic)

print("Synthetic sample created:")
print(f"  ID: {sample.sample_id}")
print(f"  Source: {sample.source}")
print(f"  Text length: {len(sample.text)} chars")
print(f"  Image size: {sample.image.size if sample.image else None}")
print(f"  Text preview: {sample.text[:100]}...")

# Display image
if sample.image:
    plt.figure(figsize=(4, 4))
    plt.imshow(sample.image)
    plt.title("Synthetic Sample Image")
    plt.axis('off')
    plt.show()

In [ ]:
# Route the synthetic sample
result = engine.route_sample(sample)

print("\nRouter Decision:")
print(f"  Chosen model: {result.router_decision.chosen_model}")
print(f"  Inference time: {result.router_decision.inference_ms:.2f} ms")
print(f"\nProbability distribution:")
for model, prob in result.router_decision.probs.items():
    print(f"  {model:30s}: {prob:.4f}")

# Visualize probabilities
models = list(result.router_decision.probs.keys())
probs = list(result.router_decision.probs.values())

plt.figure(figsize=(10, 5))
plt.bar(models, probs)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Probability')
plt.title('Router Probability Distribution (Synthetic Sample)')
plt.tight_layout()
plt.show()

## 4. Test with Database Sample (Single)

In [ ]:
# Load a sample by ID from the test split
# Update this with an actual sample_id from your database
sample_id = "ai2d_00004_bf3d9c5fd30bf304"  # Example - replace with actual ID

try:
    result = engine.route_by_id(sample_id, split="test")
    
    print(f"Sample loaded from DB:")
    print(f"  ID: {result.sample.sample_id}")
    print(f"  Source: {result.sample.source}")
    print(f"  Split: {result.sample.metadata.get('split')}")
    print(f"  Text preview: {result.sample.text[:200]}...")
    print(f"  Label: {result.sample.label}")
    
    print(f"\nRouter Decision:")
    print(f"  Chosen model: {result.router_decision.chosen_model}")
    print(f"  Inference time: {result.router_decision.inference_ms:.2f} ms")
    print(f"  Top-3 probabilities:")
    
    sorted_probs = sorted(result.router_decision.probs.items(), key=lambda x: x[1], reverse=True)
    for i, (model, prob) in enumerate(sorted_probs[:3]):
        print(f"    {i+1}. {model:30s}: {prob:.4f}")
    
    # Check if router matches label
    if result.sample.label:
        is_correct = result.router_decision.chosen_model == result.sample.label
        print(f"\n  Router correct: {is_correct}")
    
    # Display image if available
    if result.sample.image:
        plt.figure(figsize=(6, 6))
        plt.imshow(result.sample.image)
        plt.title(f"Sample: {result.sample.sample_id}")
        plt.axis('off')
        plt.show()
    
except Exception as e:
    print(f"Error loading sample: {e}")
    print("Please update sample_id with a valid ID from your database")

## 5. Test with Database Batch

In [ ]:
# Route a batch of samples from test split
batch_size = 32

print(f"Loading {batch_size} samples from test split...")
results = engine.route_split("test", limit=batch_size)

print(f"\nRouted {len(results)} samples")

# Calculate statistics
latencies = [r.router_decision.inference_ms for r in results]
chosen_models = [r.router_decision.chosen_model for r in results]

print(f"\nLatency statistics:")
print(f"  Mean: {np.mean(latencies):.2f} ms")
print(f"  Median: {np.median(latencies):.2f} ms")
print(f"  P95: {np.percentile(latencies, 95):.2f} ms")
print(f"  P99: {np.percentile(latencies, 99):.2f} ms")

print(f"\nModel distribution:")
model_counts = pd.Series(chosen_models).value_counts()
for model, count in model_counts.items():
    print(f"  {model:30s}: {count:3d} ({count/len(results)*100:.1f}%)")

In [ ]:
# Visualize latency distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(latencies, bins=20, edgecolor='black')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.title('Router Latency Distribution')

plt.subplot(1, 2, 2)
model_counts.plot(kind='bar')
plt.xlabel('Model')
plt.ylabel('Count')
plt.title('Model Selection Distribution')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 6. Calculate Accuracy (if labels available)

In [ ]:
# Check accuracy for samples with labels
samples_with_labels = [r for r in results if r.sample.label is not None]

if samples_with_labels:
    correct = sum(
        1 for r in samples_with_labels 
        if r.router_decision.chosen_model == r.sample.label
    )
    total = len(samples_with_labels)
    accuracy = correct / total
    
    print(f"Accuracy on samples with labels:")
    print(f"  Correct: {correct} / {total}")
    print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # Per-model accuracy
    print(f"\nPer-model accuracy (when that model is the true best):")
    for model_name in cfg.router.model_name_order:
        model_samples = [
            r for r in samples_with_labels 
            if r.sample.label == model_name
        ]
        if model_samples:
            model_correct = sum(
                1 for r in model_samples 
                if r.router_decision.chosen_model == model_name
            )
            model_acc = model_correct / len(model_samples)
            print(f"  {model_name:30s}: {model_acc:.4f} ({len(model_samples)} samples)")
else:
    print("No samples with labels found in batch")

## 7. Test Multiple Synthetic Samples

In [ ]:
# Test routing multiple synthetic samples
n_synthetic = 50

print(f"Routing {n_synthetic} synthetic samples...")

synthetic_samples = [make_synthetic_sample(i, cfg.traffic) for i in range(n_synthetic)]
synthetic_results = engine.route_batch(synthetic_samples)

# Statistics
synth_latencies = [r.router_decision.inference_ms for r in synthetic_results]
synth_models = [r.router_decision.chosen_model for r in synthetic_results]

print(f"\nSynthetic batch statistics:")
print(f"  Mean latency: {np.mean(synth_latencies):.2f} ms")
print(f"  P95 latency: {np.percentile(synth_latencies, 95):.2f} ms")

print(f"\nModel distribution:")
synth_counts = pd.Series(synth_models).value_counts()
for model, count in synth_counts.items():
    print(f"  {model:30s}: {count:3d} ({count/len(synthetic_results)*100:.1f}%)")

## 8. Summary

In [ ]:
print("="*60)
print("ROUTER UNIT TEST SUMMARY")
print("="*60)

print(f"\n✓ Configuration loaded successfully")
print(f"✓ Router engine initialized")
print(f"✓ Synthetic sample routing works")
print(f"✓ Database sample routing works")
print(f"✓ Batch routing works")

print(f"\nRouter Configuration:")
print(f"  Device: {cfg.router.device}")
print(f"  Dtype: {cfg.router.dtype}")
print(f"  Models: {len(cfg.router.model_name_order)}")

print(f"\nPerformance:")
print(f"  Avg latency (DB batch): {np.mean(latencies):.2f} ms")
print(f"  Avg latency (synthetic): {np.mean(synth_latencies):.2f} ms")

if samples_with_labels:
    print(f"\nAccuracy (on labeled samples):")
    print(f"  {accuracy:.4f} ({accuracy*100:.2f}%)")

print(f"\n" + "="*60)
print("All tests completed successfully!")
print("="*60)